<a href="https://colab.research.google.com/github/Fernandateireis/dotfiles/blob/master/cbo_cod.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers torch sentence-transformers pandas numpy scikit-learn openpyxl gensim


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 79.1 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from google.colab import files
import io  # Para bytes
import warnings
warnings.filterwarnings('ignore')

In [33]:
# Mapeamento corrigido (tudo minúsculo para bater com nome_lower)
mapa_nomes = {
    'cbo_2022_familias' : 'cbo',
    'cod_2022'          : 'cod',
    'qbq'               : 'qbq',
    'cbo_2022_perfil'   : 'cbo_perfil'   # minúsculo aqui!
}

# Upload
print("Selecione os arquivos...")
uploaded = files.upload()

# Lê e nomeia direto
for nome_arq, conteudo in uploaded.items():
    nome_lower = nome_arq.lower().replace('.xlsx', '')  # remove extensão e minusculiza
    nome_var = None

    for chave, nome_curto in mapa_nomes.items():
        if chave in nome_lower:
            nome_var = nome_curto
            break

    if nome_var:
        globals()[nome_var] = pd.read_excel(io.BytesIO(conteudo))
        print(f"✅ '{nome_arq}'  →  '{nome_var}' {globals()[nome_var].shape}")
    else:
        print(f"⚠️ '{nome_arq}' não reconhecido — adicione ao mapa_nomes")

# Confirma tudo
print("\n=== PRIMEIRAS 5 LINHAS ===")
for nome_var in ['cbo', 'cod', 'qbq', 'cbo_perfil']:
    if nome_var in globals():
        df = globals()[nome_var]
        print(f"\n📁 {nome_var}: {df.shape}")
        print(df.head(5))
        print("-" * 60)
    else:
        print(f"\n❌ '{nome_var}' não carregado")

Selecione os arquivos...


Saving CBO_2022_perfil_ocupacional.xlsx to CBO_2022_perfil_ocupacional.xlsx
Saving QBQ.xlsx to QBQ.xlsx
Saving cod_2022_familias.xlsx to cod_2022_familias.xlsx
Saving cbo_2022_familias.xlsx to cbo_2022_familias.xlsx
✅ 'CBO_2022_perfil_ocupacional.xlsx'  →  'cbo_perfil' (183744, 9)
✅ 'QBQ.xlsx'  →  'qbq' (2673, 8)
✅ 'cod_2022_familias.xlsx'  →  'cod' (434, 2)
✅ 'cbo_2022_familias.xlsx'  →  'cbo' (627, 2)

=== PRIMEIRAS 5 LINHAS ===

📁 cbo: (627, 2)
   CBO_codigo                              CBO_titulo
0         101    Oficiais generais das forças armadas
1         102             Oficiais das forças armadas
2         103               Praças das forças armadas
3         201  Oficiais superiores da polícia militar
4         202            Capitães da  polícia militar
------------------------------------------------------------

📁 cod: (434, 2)
   COD_codigo                                         COD_titulo
0        1111                                      Legisladores 
1        1112   

In [36]:
# Criar colunas limpas
cbo['titulo_clean'] = (
    cbo['CBO_titulo']
    .astype(str)
    .str.strip()
    .str.lower()
)

cod['titulo_clean'] = (
    cod['COD_titulo']
    .astype(str)
    .str.strip()
    .str.lower()
)

# Remover vazios muito curtos (se houver)
cbo_valid = cbo[cbo['titulo_clean'].str.len() > 3].reset_index(drop=True)
cod_valid = cod[cod['titulo_clean'].str.len() > 3].reset_index(drop=True)

print("CBO válidos:", len(cbo_valid))
print("COD válidos:", len(cod_valid))
print("\nExemplo CBO:", cbo_valid['titulo_clean'].head())
print("Exemplo COD:", cod_valid['titulo_clean'].head())

(627, 2) Index(['CBO_codigo', 'CBO_titulo'], dtype='object')
(434, 2) Index(['COD_codigo', 'COD_titulo'], dtype='object')
CBO válidos: 627
COD válidos: 434

Exemplo CBO: 0      oficiais generais das forças armadas
1               oficiais das forças armadas
2                 praças das forças armadas
3    oficiais superiores da polícia militar
4              capitães da  polícia militar
Name: titulo_clean, dtype: object
Exemplo COD: 0                                         legisladores
1       dirigentes superiores da administração pública
2                        chefes de pequenas populações
3    dirigentes de organizações que apresentam um i...
4                   diretores gerais e gerentes gerais
Name: titulo_clean, dtype: object


Gerar embeddings em português
Usei um modelo de sentença em PT-BR (bem melhor que distância de string): SentenceTransformer
Ele funciona muito bem para PT-BR e é leve (bom para Colab)

In [37]:
# Modelo em português (frases)
model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
# Ele funciona muito bem para PT-BR e é leve (bom para Colab)

cbo_embeddings = model.encode(
    cbo_valid['titulo_clean'].tolist(),
    show_progress_bar=True
)
cod_embeddings = model.encode(
    cod_valid['titulo_clean'].tolist(),
    show_progress_bar=True
)

print("Shape CBO embeddings:", cbo_embeddings.shape)
print("Shape COD embeddings:", cod_embeddings.shape)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/20 [00:00<?, ?it/s]

Batches:   0%|          | 0/14 [00:00<?, ?it/s]

Shape CBO embeddings: (627, 384)
Shape COD embeddings: (434, 384)


Similaridade e pareamento (top 1 e top 3)
Usei  similaridade cosseno para achar os COD mais próximos de cada CBO.



Top 1 (melhor par por CBO): cosine_similarity

In [38]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Matriz de similaridade: cada linha = CBO, cada coluna = COD
sim_matrix = cosine_similarity(cbo_embeddings, cod_embeddings)
sim_matrix.shape

(627, 434)

In [39]:
# Para cada CBO, pega o COD mais parecido
melhores_matches = []

for i, row in cbo_valid.iterrows():
    sims = sim_matrix[i]
    j = np.argmax(sims)  # índice do COD com maior similaridade
    melhores_matches.append({
        'CBO_codigo': row['CBO_codigo'],
        'CBO_titulo': row['CBO_titulo'],
        'COD_codigo_match': cod_valid.iloc[j]['COD_codigo'],
        'COD_titulo_match': cod_valid.iloc[j]['COD_titulo'],
        'similaridade': sims[j]
    })

base_match_top1 = pd.DataFrame(melhores_matches)

# Filtrar por um limiar de confiança (ajustável)
base_match_top1_filtrado = base_match_top1[base_match_top1['similaridade'] >= 0.7].reset_index(drop=True)

print("Total CBO:", len(cbo_valid))
print("Matches com similaridade >= 0.7:", len(base_match_top1_filtrado))
base_match_top1_filtrado.head(10)

Total CBO: 627
Matches com similaridade >= 0.7: 562


,CBO_codigo,CBO_titulo,COD_codigo_match,COD_titulo_match,similaridade
0,101,Oficiais generais das forças armadas,110,Oficiais das forças armadas,0.949472
1,102,Oficiais das forças armadas,110,Oficiais das forças armadas,1.000000
2,103,Praças das forças armadas,110,Oficiais das forças armadas,0.891348
3,201,Oficiais superiores da polícia militar,411,Oficiais de polícia militar,0.972308
4,202,Capitães da polícia militar,411,Oficiais de polícia militar,0.936416
5,203,Tenentes da polícia militar,411,Oficiais de polícia militar,0.967671
6,211,Subtenentes e sargentos da policia militar,411,Oficiais de polícia militar,0.949546
7,212,Cabos e soldados da polícia militar,411,Oficiais de polícia militar,0.846531
8,301,Oficiais superiores do corpo de bombeiros militar,511,Oficiais de bombeiro militar,0.961135
9,302,Oficiais intermediários do corpo de bombeiros ...,511,Oficiais de bombeiro militar,0.955289


Top 3 por CBO: para revisão

In [40]:
top3_matches = []

for i, row in cbo_valid.iterrows():
    sims = sim_matrix[i]
    # índices dos 3 CODs mais semelhantes
    top3_idx = np.argsort(sims)[-3:][::-1]  # ordena e pega 3 maiores

    for rank, j in enumerate(top3_idx, start=1):
        top3_matches.append({
            'CBO_codigo': row['CBO_codigo'],
            'CBO_titulo': row['CBO_titulo'],
            'COD_codigo': cod_valid.iloc[j]['COD_codigo'],
            'COD_titulo': cod_valid.iloc[j]['COD_titulo'],
            'rank': rank,
            'similaridade': sims[j]
        })

base_match_top3 = pd.DataFrame(top3_matches)

# Se quiser só matches "razoáveis"
base_match_top3_filtrado = base_match_top3[base_match_top3['similaridade'] >= 0.6].reset_index(drop=True)

base_match_top3_filtrado.head(15)

,CBO_codigo,CBO_titulo,COD_codigo,COD_titulo,rank,similaridade
0,101,Oficiais generais das forças armadas,110,Oficiais das forças armadas,1,0.949472
1,101,Oficiais generais das forças armadas,411,Oficiais de polícia militar,2,0.844100
2,101,Oficiais generais das forças armadas,210,Graduados e praças das forças armadas,3,0.789668
3,102,Oficiais das forças armadas,110,Oficiais das forças armadas,1,1.000000
4,102,Oficiais das forças armadas,411,Oficiais de polícia militar,2,0.913075
5,102,Oficiais das forças armadas,210,Graduados e praças das forças armadas,3,0.831869
6,103,Praças das forças armadas,110,Oficiais das forças armadas,1,0.891348
7,103,Praças das forças armadas,210,Graduados e praças das forças armadas,2,0.829166
8,103,Praças das forças armadas,411,Oficiais de polícia militar,3,0.768161
9,201,Oficiais superiores da polícia militar,411,Oficiais de polícia militar,1,0.972308


Exportar para Excel

In [41]:
from google.colab import files

base_match_top1_filtrado.to_excel('cbo_cod_match_top1_embeddings.xlsx', index=False)
base_match_top3_filtrado.to_excel('cbo_cod_match_top3_embeddings.xlsx', index=False)

files.download('cbo_cod_match_top1_embeddings.xlsx')
files.download('cbo_cod_match_top3_embeddings.xlsx')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>